<a href="https://colab.research.google.com/github/jessicaromero-ctrl/NF1---First-Steps/blob/main/Pareja_4A_Clasificaci%C3%B3n_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Ingeniería de Clasificación de Riesgo en Neurofibromatosis Tipo 1 (NF1) basada en Características Clínicas: mediante Ensemble Learning**
---
* Jessica Melani Romero Lora
* Aldo Iván Hurtado Hernández

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
url_github = 'https://raw.githubusercontent.com/IvanHrdz/DataSet_Clasificaci-n-de-Riesgo-en-Neurofibromatosisneurofibromatosis/refs/heads/main/DataSet_Clasificaci%C3%B3n%20de%20Riesgo%20en%20Neurofibromatosisneurofibromatosis.csv'

In [ ]:
df = pd.read_csv(url_github, encoding='latin1')
df.head(25)

,Case Type,Tumour Case,Age of Mother,Age of Father,Age at First Diagnosis,CafÃ© au lait (CLS),Axillary Freckles,Inguinal Freckles,Lisch Nodules,Dermal Neurofibromins,Plexiform Neurofibromins,Optic Glioma,Skeletal Dysplasia,Learning Disability,Hypertension,Astrocytoma,Hamartoma,Scoliosis,Other Symptoms
0,0,1,0.482759,0.279070,0.074380,0,0,0,0,0,0,1,0,0,0,0,0,0,1
1,0,1,0.206897,0.232558,0.107438,1,1,0,0,1,0,0,0,0,0,0,0,0,1
2,1,1,0.413793,0.162791,0.140496,1,1,0,0,1,0,0,1,0,0,0,0,0,1
3,0,1,0.482759,0.348837,0.140496,1,1,0,1,0,1,0,0,0,0,0,0,0,1
4,0,1,0.379310,0.348837,0.008264,1,1,1,0,0,0,1,0,0,0,0,0,0,1
5,0,1,0.586207,0.465116,0.041322,1,0,0,0,0,0,1,0,0,0,0,0,0,0
6,0,1,0.586207,0.348837,0.057851,1,1,0,0,0,0,0,0,0,0,1,0,0,1
7,0,1,0.275862,0.348837,0.041322,1,1,1,1,0,0,1,0,0,0,0,0,0,1
8,1,1,0.275862,0.162791,0.140496,0,0,0,0,0,0,1,0,0,0,0,0,0,0
9,1,0,0.586207,0.767442,0.338843,0,0,0,0,0,0,0,0,0,0,0,0,0,0


# **Fase 1: Preprocesamiento y limpieza**
---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------
# PASO 1: Cargar dataset real desde GitHub (manejo de encoding)
# ---------------------------------------------------------

url_github = "https://raw.githubusercontent.com/IvanHrdz/dataset-uci/refs/heads/main/dataset-uci.csv"

# Intento 1: UTF-8
try:
    temp_df = pd.read_csv(url_github, encoding="utf-8")
except UnicodeDecodeError:
    print("⚠ Archivo no está en UTF-8, probando Latin-1...")
    temp_df = pd.read_csv(url_github, encoding="latin1")

# Si detecta el diccionario de datos, saltamos filas de metadatos
# Basado en el output de df.head(25) de la celda anterior, el metadato va hasta la fila 18 (19 filas en total).
# La cabecera de los datos reales debería estar en la fila 19 (índice 19).
if "Variable Name" in temp_df.columns:
    print("⚠ Archivo detectado como diccionario, intentando saltar filas de metadatos...")
    df = pd.read_csv(url_github, encoding="latin1", skiprows=19)
else:
    # Si 'Variable Name' no está en las columnas, asumimos que ya es el dataset de datos.
    df = temp_df

# Normalize column names immediately after loading to handle potential whitespace issues
df.columns = df.columns.str.strip()

# --- DEBUG: Print columns to verify target variable name ---
print("Columnas del DataFrame después de stripping:")
print(df.columns)
# ---------------------------------------------------------

# ---------------------------------------------------------
# PASO 2: Limpieza inicial
# ---------------------------------------------------------

cols_basura = [c for c in df.columns if "Unnamed" in c]
if cols_basura:
    df = df.drop(columns=cols_basura)

df.dropna(how="all", inplace=True)
df.dropna(axis=1, how="all", inplace=True)

# ---------------------------------------------------------
# PASO 3: Manejo de valores faltantes
# ---------------------------------------------------------

for col in df.columns:
    if df[col].dtype in ["int64", "float64"]:
        df[col].fillna(df[col].median(), inplace=True)
    else:
        df[col].fillna(df[col].mode()[0], inplace=True)

# ---------------------------------------------------------
# PASO 4: Variables binarias seguras
# ---------------------------------------------------------

binary_cols = [
    col for col in df.columns
    if df[col].nunique() == 2 and not df[col].dtype == "object"
]

if binary_cols:
    df[binary_cols] = df[binary_cols].astype(int)

# ---------------------------------------------------------
# PASO 5: Normalizar edades (si existen)
# ---------------------------------------------------------

age_cols = ['Age of Mother', 'Age of Father', 'Age at First Diagnosis']
age_cols = [c for c in age_cols if c in df.columns]

if age_cols:
    scaler = StandardScaler()
    df[age_cols] = scaler.fit_transform(df[age_cols])

# ---------------------------------------------------------
# PASO 6: Separar X y y
# ---------------------------------------------------------

if "Case Type" not in df.columns:
    raise ValueError("ERROR: La columna 'Case Type' no existe en el dataset.")

y = df["Case Type"]
X = df.drop(columns=["Case Type"])

# ---------------------------------------------------------
# PASO 7: Train/Test split
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ---------------------------------------------------------
# PASO 8: Exportar dataset limpio
# ---------------------------------------------------------

df.to_csv("dataset_limpio.csv", index=False, encoding="utf-8")
df.to_excel("dataset_limpio.xlsx", index=False)

# ---------------------------------------------------------
# PASO 9: Mensaje final
# ---------------------------------------------------------

print("ATENCIÓN: Dataset real cargado correctamente.")
print("Limpieza completada. El dataset está listo para la Fase 2.")
print(f"Número total de variables (X): {X.shape[1]}")

⚠ Archivo no está en UTF-8, probando Latin-1...
⚠ Archivo detectado como diccionario, intentando saltar filas de metadatos...
Columnas del DataFrame después de stripping:
Index(['Other Symptoms', 'Feature', 'Binary', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', 'No'],
      dtype='object')


ValueError: ERROR: La columna 'Case Type' no existe en el dataset.

# **Fase 2: Feature Engineering**
---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Cargar dataset limpio ya preprocesado
df = pd.read_excel("dataset_limpio.xlsx")

# Separar variables predictoras (X) y variable objetivo (y)
X = df.drop('Case Type', axis=1)
y = df['Case Type']

# Dividir en entrenamiento y prueba (sin fuga de datos)
# La selección de características y el escalado SIEMPRE deben hacerse SOLO con el set de entrenamiento
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,       # mantiene balance de clases
    random_state=42
)

# Escalado (necesario para LASSO y otros modelos sensibles)
scaler = StandardScaler()

# Ajustar en el entrenamiento y transformar
X_train_scaled = scaler.fit_transform(X_train)

# Transformar el test sin volver a ajustar
X_test_scaled = scaler.transform(X_test)

**Método 1. Regularización para selección lineal:**

```
# Iván
```


---

In [ ]:
# LIMPIEZA Y PROCESAMIENTO DEL DATASET UCI
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Cargar el archivo
df = pd.read_excel("dataset-uci.xlsx")

# Eliminar columna irrelevante
df = df.drop(columns=["Unnamed: 0"])

# Manejo de valores faltantes
#    - Numéricos → mediana
#    - Binarios/categorías → moda
for col in df.columns:
    if df[col].dtype in ["int64", "float64"]:
        df[col].fillna(df[col].median(), inplace=True)
    else:
        df[col].fillna(df[col].mode()[0], inplace=True)

# Asegurar que las variables binarias sean enteros
binary_cols = [col for col in df.columns if df[col].nunique() == 2]
df[binary_cols] = df[binary_cols].astype(int)

# Normalización de variables numéricas
num_cols = ["Age of Mother", "Age of Father", "Age at First Diagnosis"]

scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# Separación de variables predictoras y target
X = df.drop(columns=["Case Type"])
y = df["Case Type"]

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Exportar dataset limpio a CSV y Excel
df.to_csv("dataset_limpio.csv", index=False)
df.to_excel("dataset_limpio.xlsx", index=False)

print("Limpieza completada y archivos exportados.")

**Método 2: Importancia de Características por Árboles (Random Forest)**


```
# Iván
```



In [ ]:
# IMPORTANCIA DE FEATURES RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Entrenar un modelo RF (sin fine tuning)
rf_selector = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    max_depth=None
)

rf_selector.fit(X_train, y_train)

# Extraer importancia de variables
rf_importance = pd.Series(
    rf_selector.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("=== IMPORTANCIAS ORDENADAS POR RANDOM FOREST ===")
print(rf_importance)

# Selección automática:
# Features que explican el 90% de importancia acumulada
rf_cum_importance = rf_importance.cumsum()
rf_selected_features = rf_cum_importance[rf_cum_importance <= 0.90].index.tolist()

# Asegurar que se incluye la siguiente feature si falta para llegar al 90%
if len(rf_selected_features) < len(rf_importance):
    rf_selected_features.append(rf_importance.index[len(rf_selected_features)])

print("\n=== FEATURES SELECCIONADAS POR RANDOM FOREST (90% Acc) ===")
print(rf_selected_features)

# (Opcional) Intersección con LASSO
final_features = list(set(selected_lasso).intersection(set(rf_selected_features)))

print("\n=== SUBCONJUNTO FINAL DE FEATURES (LASSO ∩ RF) ===")
print(final_features)

# **Fase 3: Modelado y validación**
---


```
# Mel
```



#**Fase 4: Métricas**
---


```
# Mel
```

